# Clustering Burn Calculation

**Workflow — run cells top to bottom:**
1. **Step 1** — Select hub(s): auto from BigQuery dropdown **or** type IDs manually
2. **Step 2** — Set date range (how many days back to query)
3. **Step 3** — Upload the Live CSV (cluster polygons) and an optional pincode boundaries CSV
4. **Step 4** — Run AWB query → assign clusters → compute P&L report

## Imports & BigQuery setup

In [ ]:
import io
import pandas as pd
from shapely.geometry import Point
from google.cloud import bigquery
from shapely.wkt import loads as load_wkt
from shapely.prepared import prep
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display

pd.set_option('display.max_columns', None)
client = bigquery.Client(project='bi-team-400508')
print('BigQuery client initialised.')

---
## Step 1 — Select Hub(s)

Hub IDs are pulled live from BigQuery.  
Choose **Auto** to pick from the list, or **Manual** to type numeric IDs directly.

In [ ]:
print('Fetching hub list from BigQuery...')
_hub_df = client.query("""
    SELECT id, name
    FROM `data-warehousing-391512.ecommerce.ecommerce_hub`
    ORDER BY name
""").to_dataframe()
print(f'  {len(_hub_df):,} hubs loaded.')

_hub_opts = [
    (f"{r['name']}  (ID: {r['id']})", int(r['id']))
    for _, r in _hub_df.iterrows()
]

# ── Mode toggle ──────────────────────────────────────────────────────────────
mode_radio = widgets.RadioButtons(
    options=['Auto — select from dropdown', 'Manual — type hub IDs'],
    value='Auto — select from dropdown',
    description='Hub source:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(margin='0 0 12px 0'),
)

# ── Auto: multi-select list ───────────────────────────────────────────────────
hub_selector = widgets.SelectMultiple(
    options=_hub_opts,
    description='Hub(s):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='540px', height='160px'),
)
_auto_hint = widgets.HTML(
    '<span style="color:#666;font-size:12px">'
    'Hold <kbd>Ctrl</kbd> (Windows) / <kbd>Cmd</kbd> (Mac) to select multiple.</span>'
)
_auto_box = widgets.VBox([hub_selector, _auto_hint])

# ── Manual: text input ────────────────────────────────────────────────────────
manual_hub_input = widgets.Textarea(
    placeholder='e.g.  15344, 12345, 67890',
    description='Hub IDs:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='540px', height='70px'),
)
_manual_hint = widgets.HTML(
    '<span style="color:#666;font-size:12px">Comma-separated numeric hub IDs.</span>'
)
_manual_box = widgets.VBox([manual_hub_input, _manual_hint])
_manual_box.layout.display = 'none'

def _on_hub_mode(change):
    if change['new'].startswith('Manual'):
        _auto_box.layout.display = 'none'
        _manual_box.layout.display = ''
    else:
        _auto_box.layout.display = ''
        _manual_box.layout.display = 'none'

mode_radio.observe(_on_hub_mode, names='value')
display(mode_radio, _auto_box, _manual_box)

---
## Step 2 — Set Date Range

How many days back from today should the AWB query cover?  
Default is **30 days**. Use the slider **or** type a number directly.

In [ ]:
days_slider = widgets.IntSlider(
    value=30, min=1, max=90, step=1,
    description='Days back:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px'),
    continuous_update=False,
)
days_text = widgets.BoundedIntText(
    value=30, min=1, max=90,
    description='or type:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='160px'),
)

# Keep slider and text in sync
widgets.jslink((days_slider, 'value'), (days_text, 'value'))

display(
    widgets.HTML('<b>Query window (days before today):</b>'),
    widgets.HBox([days_slider, days_text]),
    widgets.HTML('<span style="color:#666;font-size:12px">Range: 1 – 90 days.</span>'),
)

---
## Step 3 — Upload CSV Files

| File | Purpose | Required? |
|---|---|---|
| **Live CSV** | Cluster polygon boundaries (`WKT`, `name`, `description`) | Falls back to `Live5.csv` if not uploaded |
| **Pincode CSV** | Pincode-level cluster boundaries (`pincode`, `description`) | Optional — uses built-in defaults if skipped |

In [ ]:
live_csv_upload = widgets.FileUpload(
    accept='.csv', multiple=False,
    description='Live CSV',
    button_style='primary',
    layout=widgets.Layout(width='210px'),
)
pincode_csv_upload = widgets.FileUpload(
    accept='.csv', multiple=False,
    description='Pincode CSV',
    layout=widgets.Layout(width='210px'),
)

live_status    = widgets.HTML('<span style="color:#888">No file — will fall back to <code>Live5.csv</code>.</span>')
pincode_status = widgets.HTML('<span style="color:#888">No file — built-in pincode mapping will be used.</span>')

def _upload_name(uploader):
    """Return filename string; works across ipywidgets 7 and 8."""
    try:
        return uploader.value[0]['name']          # ipywidgets >= 8 (tuple of dicts)
    except (TypeError, IndexError, KeyError):
        return list(uploader.value.keys())[0]     # ipywidgets 7 (dict keyed by filename)

def _on_live_upload(change):
    if live_csv_upload.value:
        live_status.value = (
            f'<span style="color:green">✅ Loaded: <b>{_upload_name(live_csv_upload)}</b></span>'
        )

def _on_pincode_upload(change):
    if pincode_csv_upload.value:
        pincode_status.value = (
            f'<span style="color:green">✅ Loaded: <b>{_upload_name(pincode_csv_upload)}</b></span>'
        )

live_csv_upload.observe(_on_live_upload, names='value')
pincode_csv_upload.observe(_on_pincode_upload, names='value')

display(
    widgets.HTML('<b>Cluster polygons (Live CSV):</b>'),
    live_csv_upload, live_status,
    widgets.HTML('<br><b>Pincode boundaries CSV (optional):</b>'),
    pincode_csv_upload, pincode_status,
)

---
## Preview — Confirm Selections Before Querying

Run this cell to verify your hub(s), date range, and files **before** hitting BigQuery.

In [ ]:
from IPython.display import HTML as _HTML

def _preview_selections():
    # ── Resolve hub IDs and names ────────────────────────────────────────────
    if mode_radio.value.startswith('Manual'):
        raw  = manual_hub_input.value.strip()
        _ids = [int(x.strip()) for x in raw.split(',') if x.strip().isdigit()] if raw else []
        hub_rows = [
            (hid,
             _hub_df.loc[_hub_df['id'] == hid, 'name'].values[0]
             if hid in _hub_df['id'].values else '(name not found)')
            for hid in _ids
        ]
        source_label = 'Manual input'
    else:
        hub_rows = [
            (hid, _hub_df.loc[_hub_df['id'] == hid, 'name'].values[0])
            for hid in hub_selector.value
        ]
        source_label = 'Dropdown (BigQuery)'

    days      = int(days_slider.value)
    date_from = (pd.Timestamp.today().normalize() - pd.Timedelta(days=days)).strftime('%Y-%m-%d')
    date_to   = pd.Timestamp.today().strftime('%Y-%m-%d')

    # ── File labels ──────────────────────────────────────────────────────────
    def _fname(uploader, fallback_msg):
        if uploader.value:
            try:    name = uploader.value[0]['name']          # ipywidgets >= 8
            except: name = list(uploader.value.keys())[0]    # ipywidgets 7
            return f"<b style='color:green'>{name}</b>"
        return f"<i style='color:#999'>{fallback_msg}</i>"

    live_label    = _fname(live_csv_upload,    'Not uploaded — will use Live5.csv')
    pincode_label = _fname(pincode_csv_upload, 'Not uploaded — built-in defaults')

    # ── Hub table rows ───────────────────────────────────────────────────────
    hub_rows_html = ''.join(
        f'<tr><td style="padding:5px 14px;font-weight:bold;color:#1a56db">{hid}</td>'
        f'<td style="padding:5px 14px">{name}</td></tr>'
        for hid, name in hub_rows
    ) or '<tr><td colspan="2" style="color:red;padding:5px 14px">⚠ No hubs selected!</td></tr>'

    total_hubs = len(hub_rows)

    html = f"""
    <div style="font-family:sans-serif;border:1.5px solid #99ccff;border-radius:10px;
                padding:20px 26px;max-width:660px;background:#f0f7ff;">
      <h3 style="margin:0 0 16px;color:#1e3a5f">✅ Query Preview</h3>

      <div style="margin-bottom:14px">
        <span style="color:#555">Hub source:</span>
        <b style="margin-left:6px">{source_label}</b>
        &nbsp;·&nbsp;
        <b style="color:#1a56db">{total_hubs}</b> hub(s) selected
      </div>

      <table style="border-collapse:collapse;width:100%;margin-bottom:16px;
                    border:1px solid #bcd4f5;border-radius:6px;overflow:hidden">
        <thead>
          <tr style="background:#99ccff">
            <th style="padding:6px 14px;text-align:left;font-size:13px">Hub ID</th>
            <th style="padding:6px 14px;text-align:left;font-size:13px">Hub Name</th>
          </tr>
        </thead>
        <tbody style="background:#fff">{hub_rows_html}</tbody>
      </table>

      <div style="margin-bottom:12px">
        <span style="color:#555">Date range:</span>
        <span style="font-size:20px;font-weight:bold;color:#1e3a5f;margin:0 6px">{days}</span>
        days
        &nbsp;<span style="color:#888;font-size:13px">({date_from} → {date_to})</span>
      </div>

      <div style="border-top:1px solid #bcd4f5;padding-top:12px;font-size:13px">
        <div><span style="color:#555;width:110px;display:inline-block">Live CSV:</span> {live_label}</div>
        <div style="margin-top:4px"><span style="color:#555;width:110px;display:inline-block">Pincode CSV:</span> {pincode_label}</div>
      </div>
    </div>
    """
    display(_HTML(html))

_preview_selections()

---
## Step 4 — Fetch AWB Data

Reads hub selection (Step 1) and date range (Step 2), then runs the BigQuery query.

In [ ]:
# ── Resolve hub IDs ──────────────────────────────────────────────────────────
def _get_hub_ids():
    if mode_radio.value.startswith('Manual'):
        raw = manual_hub_input.value.strip()
        if not raw:
            raise ValueError('Manual mode selected but no hub IDs entered.')
        ids = [int(x.strip()) for x in raw.split(',') if x.strip().isdigit()]
        if not ids:
            raise ValueError('No valid numeric hub IDs found in manual input.')
        return ids
    selected = hub_selector.value
    if not selected:
        raise ValueError('No hubs selected. Pick at least one from the dropdown.')
    return list(selected)

hub_ids     = _get_hub_ids()
hub_ids_str = ', '.join(str(i) for i in hub_ids)
days_back   = int(days_slider.value)

print(f'Hub ID(s)  : {hub_ids_str}')
print(f'Date range : last {days_back} day(s)')
print()

# ── AWB query (hub IDs and days injected dynamically) ────────────────────────
Awb = f'''
WITH awb_data AS (

  SELECT
    sg.order_date, sg.rider_id, sg.hub_id, sg.pincode, sg.pincode_category,
    sg.order_id                 AS fwd_del_awb_number,
    edp.delivery_latitude       AS lat,
    edp.delivery_longitude      AS long,
    ROW_NUMBER() OVER (PARTITION BY sg.rider_id ORDER BY edp.update_timestamp) AS row_num
  FROM `data-warehousing-391512.smaug_dataengine.data_engine_orderleveldata` sg
  LEFT JOIN `data-warehousing-391512.ecommerce.ecommerce_deliveryrequest` edr
    ON edr.awb_number = sg.order_id
   AND edr.last_updated > CURRENT_DATE - INTERVAL {days_back} DAY
  LEFT JOIN `data-warehousing-391512.ecommerce.ecommerce_deliveryrequestproof` edp
    ON edr.id = edp.delivery_request_id
   AND edp.update_timestamp > CURRENT_DATE - INTERVAL {days_back} DAY
  WHERE sg.order_date > CURRENT_DATE - INTERVAL {days_back} DAY
    AND sg.order_category = 1
    AND ecom_request_type IN (1)
    AND sg.order_status   IN (1)
    AND sg.order_tag      IN (0, 1, 14)
    AND edr.client_id NOT IN (
          5,18,60,61,67,68,102,354,552,557,715,
          818,862,875,11,996,1579,1575,1819,2063,2253
        )
    AND sg.hub_id IN ({hub_ids_str})

  UNION ALL

  SELECT
    sg.order_date, sg.rider_id, sg.hub_id, sg.pincode, sg.pincode_category,
    sg.order_id              AS fwd_del_awb_number,
    epp.pickup_latitude      AS lat,
    epp.pickup_longitude     AS long,
    ROW_NUMBER() OVER (PARTITION BY sg.rider_id ORDER BY epp.update_timestamp) AS row_num
  FROM `data-warehousing-391512.smaug_dataengine.data_engine_orderleveldata` sg
  LEFT JOIN `data-warehousing-391512.ecommerce.pickup_pickuprequestproof` epp
    ON sg.order_id = epp.pickup_request_id
   AND epp.update_timestamp > CURRENT_DATE - INTERVAL {days_back} DAY
  WHERE sg.order_date > CURRENT_DATE - INTERVAL {days_back} DAY
    AND sg.order_category = 1
    AND ecom_request_type IN (5)
    AND sg.order_status   IN (2, 3)
    AND sg.order_tag      IN (0, 1, 14)
    AND sg.hub_id IN ({hub_ids_str})
)

SELECT
  order_date,
  rider_id,
  eh.name AS hub,
  ad.pincode,
  CONCAT(CAST('P' AS STRING), ad.pincode_category) AS payment_category,
  fwd_del_awb_number,
  COALESCE(lat,  FIRST_VALUE(lat)  OVER (
      PARTITION BY rider_id ORDER BY row_num
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS lat,
  COALESCE(long, FIRST_VALUE(long) OVER (
      PARTITION BY rider_id ORDER BY row_num
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS long
FROM awb_data ad
LEFT JOIN `data-warehousing-391512.ecommerce.ecommerce_hub` eh
  ON ad.hub_id = eh.id
'''

print('Running BigQuery...')
Awn_number_with_latlong = client.query(Awb).to_dataframe()
print(f'Query complete — {len(Awn_number_with_latlong):,} rows fetched.')
Awn_number_with_latlong

---
## Step 5 — Assign Clusters

Loads the uploaded Live CSV (or `Live5.csv` as fallback) and matches each AWB point to a cluster polygon.

In [ ]:
# ── Helper: read an uploaded file into a DataFrame ───────────────────────────
def _read_uploaded_csv(uploader):
    if not uploader.value:
        return None
    try:
        # ipywidgets >= 8: value is a tuple/list of dicts
        file_info = uploader.value[0]
        content   = file_info['content']
    except (TypeError, IndexError, KeyError):
        # ipywidgets 7: value is a dict keyed by filename
        content = list(uploader.value.values())[0]['content']
    # content may be memoryview (ipywidgets 8) or bytes (ipywidgets 7)
    if isinstance(content, memoryview):
        content = content.tobytes()
    return pd.read_csv(io.BytesIO(bytes(content)))


# ── Load cluster polygons ─────────────────────────────────────────────────────
def load_clusters(uploader_or_path):
    if hasattr(uploader_or_path, 'value'):
        df = _read_uploaded_csv(uploader_or_path)
        if df is None:
            print("⚠️  No file uploaded — falling back to 'Live5.csv'")
            df = pd.read_csv('Live5.csv')
    else:
        df = pd.read_csv(uploader_or_path)

    clusters, skipped = [], []
    for idx, row in df.iterrows():
        try:
            wkt_str = str(row['WKT']).strip()
            if 'POLYGON ()' in wkt_str:
                skipped.append((idx, row['name'], 'Empty polygon')); continue
            if wkt_str.startswith('GEOMETRYCOLLECTION'):
                skipped.append((idx, row['name'], 'GeometryCollection not supported')); continue
            polygon = load_wkt(wkt_str)
            if polygon.is_empty:
                skipped.append((idx, row['name'], 'Parsed empty geometry')); continue
            clusters.append({
                'prepared':    prep(polygon),
                'polygon':     polygon,
                'name':        row['name'],
                'description': row['description'],
            })
        except Exception as e:
            skipped.append((idx, row['name'], str(e)))

    if skipped:
        print(f'\n⚠️  Skipped {len(skipped)} invalid rows:')
        for s in skipped:
            print(f'  Row {s[0]} | {s[1]} | {s[2]}')
    print(f'\n✅ Loaded {len(clusters)} clusters')
    return clusters


# ── Load pincode map ──────────────────────────────────────────────────────────
def _load_pincode_map(uploader):
    """Return {pincode: description}. Falls back to built-in defaults."""
    df = _read_uploaded_csv(uploader)
    if df is not None:
        print(f'  Pincode CSV loaded: {len(df)} rows')
        return dict(zip(df['pincode'].astype(int), df['description'].astype(str)))
    print('  Using built-in pincode fallback mapping.')
    return {580011: 'C4', 203209: 'C8', 282009: 'C6',
            584128: 'C2', 110074: 'C2', 800001: 'C0'}


# ── Geometry lookup ───────────────────────────────────────────────────────────
def get_cluster_for_point(lat, lon, clusters):
    if pd.isnull(lat) or pd.isnull(lon):
        return None, None
    point = Point(lon, lat)
    for c in clusters:
        if c['prepared'].contains(point):
            return c['name'], c['description']
    return None, None


# ── Process AWB rows ──────────────────────────────────────────────────────────
def process_awb_data(awb_df, clusters, pincode_map):
    results = []
    for row in tqdm(awb_df.itertuples(index=False), total=len(awb_df)):
        name, description = get_cluster_for_point(row.lat, row.long, clusters)
        if not name and row.pincode in pincode_map:
            name        = 'Previous mapping'
            description = pincode_map[row.pincode]
        results.append({
            'order_date':       row.order_date,
            'awb_number':       row.fwd_del_awb_number,
            'rider_id':         row.rider_id,
            'pincode':          row.pincode,
            'payment_category': row.payment_category,
            'hub':              row.hub,
            'lat':              row.lat,
            'long':             row.long,
            'cluster_name':     name,
            'description':      description,
        })
    return pd.DataFrame(results)

In [ ]:
clusters     = load_clusters(live_csv_upload)
pincode_map  = _load_pincode_map(pincode_csv_upload)
final_result_df = process_awb_data(Awn_number_with_latlong, clusters, pincode_map)
final_result_df

In [ ]:
final_result_df.to_csv('final_result_df.csv', index=False)
print('Saved → final_result_df.csv')

---
## Step 6 — Burn Calculation & P&L Report

In [ ]:
payment_mapping = {
    'P1': 0,   'P2': 1,   'P3': 2,   'P4': 4,   'P5': 6,
    'P6': 8,   'P7': 9,   'P8': 10,  'P9': 10,  'P10': 10,
    'P11': 11, 'P12': 12, 'P13': 13, 'P14': 14, 'P15': 15,
    'P16': 16, 'P17': 17, 'P18': 18, 'P19': 19, 'P20': 20,
    'P21': 21, 'P22': 22, 'P23': 23, 'P24': 24, 'P25': 25,
    'P26': 26, 'P27': 27, 'P28': 28, 'P29': 29, 'P30': 30,
    # Cyrillic half-step variants (as in original)
    '\u0420\u04071': 0.5, '\u0420\u04072': 1.5, '\u0420\u04073': 2.5,
    '\u0420\u040734': 3.0, '\u0420\u04075': 3.5, '\u0420\u04076': 4.5,
    '\u0420\u04077': 5.0, '\u0420\u04078': 5.5, '\u0420\u04079': 6.5,
    'P40': 7.0, '\u0420341': 7.5,
}

description_mapping = {
    'C1': 0,   'C2': 0.5, 'C3': 1,   'C4': 1.5, 'C5': 2,
    'C6': 2.5, 'C7': 3,   'C8': 3.5, 'C9': 4,   'C10': 4.5,
    'C11': 5,  'C12': 6,  'C13': 7,  'C14': 8,  'C15': 9,
    'C16': 10, 'C17': 11, 'C18': 12, 'C19': 13, 'C20': 15,
}

final_result_df['Pin_Pay'] = (
    final_result_df['payment_category'].map(payment_mapping).fillna(0)
)
final_result_df['Clustering_payout'] = (
    final_result_df['description'].map(description_mapping)
    .fillna(final_result_df['Pin_Pay'])
)
final_result_df['P & L']   = final_result_df['Pin_Pay'] - final_result_df['Clustering_payout']
final_result_df['Saving']  = final_result_df['P & L'].apply(lambda x: x  if x > 0 else 0)
final_result_df['Burning'] = final_result_df['P & L'].apply(lambda x: -x if x < 0 else 0)

_fin_cols = ['Pin_Pay', 'Clustering_payout', 'Saving', 'Burning', 'P & L']
final_result_df = (
    final_result_df[
        ~(final_result_df[_fin_cols] == 0).all(axis=1)
    ].reset_index(drop=True)
)
final_result_df

In [ ]:
pivot = final_result_df.pivot_table(
    index='hub',
    values=['Pin_Pay', 'Clustering_payout', 'Saving', 'Burning', 'P & L'],
    aggfunc='sum',
    margins=True,
    margins_name='Grand Total',
).reset_index()

report = pivot.rename(columns={
    'Pin_Pay':           'Expt_Pincode_Pay',
    'Clustering_payout': 'Cluster_Payout',
})

_num_cols = ['Expt_Pincode_Pay', 'Cluster_Payout', 'Saving', 'Burning', 'P & L']
report[_num_cols] = report[_num_cols].apply(pd.to_numeric, errors='coerce')
report['P & L %'] = (report['P & L'] / report['Expt_Pincode_Pay'] * 100).round(2)

grand_total  = report[report['hub'] == 'Grand Total']
rest         = report[report['hub'] != 'Grand Total'].sort_values('P & L', ascending=False)
report_final = (
    pd.concat([grand_total, rest])
    [['hub'] + _num_cols + ['P & L %']]
    .set_index('hub')
    .rename_axis(index=None)
)

(
    report_final.style
    .map(lambda v: 'color: green' if isinstance(v, (int, float)) and v > 0
         else ('color: red' if isinstance(v, (int, float)) and v < 0 else ''),
         subset=['P & L', 'P & L %'])
    .format({
        'Expt_Pincode_Pay': '{:,.0f}',
        'Cluster_Payout':   '{:,.0f}',
        'Saving':           '{:,.0f}',
        'Burning':          '{:,.0f}',
        'P & L':            '{:,.0f}',
        'P & L %':          '{:.2f}%',
    })
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([{'selector': 'th', 'props': [
        ('text-align', 'center'),
        ('font-weight', 'bold'),
        ('background-color', '#99ccff'),
    ]}])
)